<a href="https://colab.research.google.com/github/lricci03/Hands-on-ML/blob/main/CFM2024-new/create-submission.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Create submission

This is a template to create the submission file for the challenge.

The submission is a csv file containing the prediction for the test set.

Workflow:

Once we have finished model selection:
- recompute normalization statistics on the full training dataset;
- normalize the full training set with those statistics;
- retrain the chosen architecture on all training data;
- normalize the test set using the same training-set mean/std;
- predict on test.

Given a model, that is the best performing on the train-validation split, we train it on the whole train set (without train-validation split) and then we use it to predict the test set.

If the data were normalized, we need to normalize them using the mean and standard deviation of the **entire train set**, while in the previous step they were computed only on the train set without the validation set.

Then, we need to use these mean and standard deviation to normalize the test set as well.

In [24]:
import torch
import torch.nn as nn

if torch.cuda.is_available():
  device = 'cuda'
elif torch.backends.mps.is_available():
  device = 'mps'
else:
  device = 'cpu'

In [1]:
import pandas as pd

In [2]:
from google.colab import drive

# drive.flush_and_unmount()
drive.mount("/content/drive")

Mounted at /content/drive


## Train set & Targets

In [3]:
df = pd.read_parquet("/content/drive/MyDrive/Challengedata/Xtrain.parquet")

df_num = df[['obs_id','price', 'bid', 'ask', 'bid_size', 'ask_size', 'flux', 'action', 'venue']]
df_num['venue'] = df_num['venue'].apply(str)
df_num = pd.get_dummies(df_num, dtype=float)

/tmp/ipykernel_3525/1214356989.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_num['venue'] = df_num['venue'].apply(str)


In [4]:
import gc

del df
gc.collect()

0

In [5]:
targets = pd.read_csv("/content/drive/MyDrive/Challengedata/y_train_or6m3Ta.csv")

In [6]:
targets = targets.set_index("obs_id")
df_num = df_num.set_index("obs_id")

In [7]:
import torch
import torch.nn as nn

X_seq = torch.tensor(df_num.values, dtype=torch.float32)
X_seq = X_seq.reshape(-1, 100, 15)
y = torch.tensor(targets.values.squeeze(), dtype=torch.long)

In [8]:
del df_num, targets
gc.collect()

97

### Train set engineered features

In [9]:
df_eng = pd.read_csv("/content/drive/MyDrive/Challengedata/eng_features.csv")

In [ ]:
df_eng = df_eng.set_index("obs_id")

In [ ]:
X_eng = torch.tensor(df_eng.values, dtype=torch.float32)

In [ ]:
mean = X_eng.mean(dim=0, keepdim=True)
std = X_eng.std(dim=0, keepdim=True)

# avoid division by zero for constant features
std[std == 0] = 1.0

X_eng_std = (X_eng - mean) / std

## Test set

In [10]:
test = pd.read_parquet("/content/drive/MyDrive/Challengedata/Xtest.parquet")

In [11]:
test = test[['obs_id','price', 'bid', 'ask', 'bid_size', 'ask_size', 'flux', 'action', 'venue']]
test['venue'] = test['venue'].apply(str)
test = pd.get_dummies(test, dtype=float)
test = test.set_index('obs_id')

In [12]:
X_test = torch.tensor(test.values, dtype=torch.float32)
X_test = X_test.reshape(-1, 100, 15)

In [13]:
del test
gc.collect()

0

### Test set engineered features

TO DO

## Datasets and DataLoaders

Dataset to combine sequential and engineered features

In [ ]:
from torch.utils.data import Dataset
import torch

class StockDataset(Dataset):
    def __init__(self, X_seq, X_eng, y):
        self.X_seq = X_seq
        self.X_eng = X_eng
        self.y = y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return (
            self.X_seq[idx],
            self.X_eng[idx],
            self.y[idx]
        )

In [ ]:
train_dataset = StockDataset(X_seq, X_eng, y)

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

Sequential features only

In [14]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset_seq = TensorDataset(X_seq, y)
train_loader_seq = DataLoader(train_dataset_seq, batch_size=64, shuffle=True)

Test set

In [15]:
test_dataset = TensorDataset(X_test)
test_loader = DataLoader(test_dataset, batch_size=256)

## Train function

We don't need the validation and the best model logic we used during model selection.

We train on the number of epochs we selected during model selection.

Train function for sequential data only

In [17]:
%pip install -q torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 8.2 MB/s eta 0:00:00


In [19]:
import torchmetrics


def train_full(model, optimizer, loss_fn, metric, train_loader, n_epochs):

    history = {
        "train_losses": [],
        "train_metrics": []
    }

    for epoch in range(n_epochs):
        total_loss = 0.0
        metric.reset()
        model.train()

        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)

            y_pred = model(X_batch)
            loss = loss_fn(y_pred, y_batch)

            total_loss += loss.item()

            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            y_class = y_pred.detach().argmax(dim=1)
            metric.update(y_class, y_batch)

        train_loss = total_loss / len(train_loader)
        train_metric = metric.compute().item()

        history["train_losses"].append(train_loss)
        history["train_metrics"].append(train_metric)

        print(
            f"Epoch {epoch + 1}/{n_epochs}, "
            f"train loss: {train_loss:.4f}, "
            f"train metric: {train_metric:.4f}"
        )

    return history

Train for joint model

In [ ]:
def train_full_joint(model, optimizer, loss_fn, metric, train_loader, n_epochs):

    history = {
        "train_losses": [],
        "train_metrics": []
    }

    for epoch in range(n_epochs):
        total_loss = 0.0
        metric.reset()
        model.train()

        for X_seq_batch, X_eng_batch, y_batch in train_loader:
            X_seq_batch = X_seq_batch.to(device)
            X_eng_batch = X_eng_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()

            y_pred = model(X_seq_batch, X_eng_batch)
            loss = loss_fn(y_pred, y_batch)

            total_loss += loss.item()

            loss.backward()
            optimizer.step()

            y_class = y_pred.detach().argmax(dim=1)
            metric.update(y_class, y_batch)

        train_loss = total_loss / len(train_loader)
        train_metric = metric.compute().item()

        history["train_losses"].append(train_loss)
        history["train_metrics"].append(train_metric)

        print(
            f"Epoch {epoch + 1}/{n_epochs}, "
            f"train loss: {train_loss:.4f}, "
            f"train metric: {train_metric:.4f}"
        )

    return history

## Predict the data

### Model

The best model is a Deep GRU (2 gru layers) that we trained on the sequential features only (so not joint), and the features are not normalized.

It is saved as `deep_gru_model_128.pt`

In [21]:
class DeepGRUModel(nn.Module):
  def __init__(self, input_size, hidden_size, output_size):
    super().__init__()
    self.hidden_size = hidden_size
    self.rnn = nn.GRU(input_size, hidden_size, num_layers=2, batch_first=True)
    self.output = nn.Linear(hidden_size, output_size)

  def forward(self,X):
    outputs, last_state = self.rnn(X)
    return self.output(outputs[:, -1])

In [26]:
model = DeepGRUModel(input_size=15, hidden_size=128, output_size=24).to(device)


model.load_state_dict(
    torch.load("/content/drive/MyDrive/Challengedata/deep_gru_model_128.pt")
)

RuntimeError: Attempting to deserialize object on a CUDA device but torch.cuda.is_available() is False. If you are running on a CPU-only machine, please use torch.load with map_location=torch.device('cpu') to map your storages to the CPU.

In [ ]:
model.eval()

predictions = []

with torch.no_grad():
    for (X_batch,) in test_loader:
        X_batch = X_batch.to(device)

        logits = model(X_batch)
        y_pred = logits.argmax(dim=1)

        predictions.extend(y_pred.cpu().numpy())

Save the predictions in csv

In [ ]:
submission = pd.DataFrame({
    "obs_id": range(len(predictions)),
    "eqt_code_cat": predictions
})

submission.to_csv(
    "/content/drive/MyDrive/Challengedata/submission.csv",
    index=False
)